# Trabalho 2: Planeamento de Trajetos
**Inteligência Artificial · UC 90498 · ESTGA, Universidade de Aveiro**

Autores: Nadia Bento e Jessica Pereira

Este notebook compara **Dijkstra** e **A\*** num planeador de trajetos em grelha 2D, com
4 e 8 direções, várias heurísticas e cenários de dificuldade diferente.

| Secção | Conteúdo |
|---|---|
| 1 | Cenários |
| 2 | Validação com o Anexo A |
| 3 | Dijkstra vs A* |
| 4 | Heurísticas e modelo de movimento |
| 5 | Heurística não admissível (Weighted A*) |
| 6 | Conclusões |


In [ ]:
import sys, math
sys.path.append("..")                      # encontra a pasta planner

import matplotlib.pyplot as plt
import pandas as pd

from planner.grid import ler_cenario
from planner.search import dijkstra, astar
from planner.heuristics import manhattan, euclidiana, chebyshev
from planner.generators import gerar_todos
from planner.experiments import correr
from planner.visualize import mostrar, grafico_expandidos, grafico_peso

# estilo limpo para todos os gráficos
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 11,
})
SEMENTE = 2026

ImportError: cannot import name 'ler_cenario' from 'planner.grid' (c:\Users\nadia\Documents\GitHub\PathFinder-AI\notebooks\..\planner\grid.py)

: 

## 1. Cenários

Usamos **6 cenários**: os 3 do Anexo A e 3 criados por nós.

| Cenário | Objetivo |
|---|---|
| A, B, C (Anexo A) | Validar a implementação (labirinto, custos variados, sem caminho) |
| Labirinto 51×51 | Muitos becos sem saída |
| Aberto 100×100 | Escalabilidade: poucos obstáculos e caminho longo |
| Custos variados | O caminho com menos passos **não** é o mais barato |

In [ ]:
gerar_todos("../scenarios/custom", seed=SEMENTE)   # mesmos ficheiros em qualquer máquina

ficheiros = {
    "A labirinto":     "../scenarios/anexo_a/A_labirinto.txt",
    "B terrenos":      "../scenarios/anexo_a/B_terrenos.txt",
    "C sem caminho":   "../scenarios/anexo_a/C_sem_caminho.txt",
    "Labirinto 51x51": "../scenarios/custom/labirinto_51x51.txt",
    "Aberto 100x100":  "../scenarios/custom/aberto_100x100.txt",
    "Custos variados": "../scenarios/custom/custos_variados.txt",
}
cenarios = {nome: ler_cenario(p) for nome, p in ficheiros.items()}

# resumo dos cenários
pd.DataFrame([{
    "Cenário": nome,
    "Dimensão": f"{g.shape[0]}×{g.shape[1]}",
    "Obstáculos (%)": round(100 * (g == 0).mean(), 1),
    "Custo máx. de célula": int(g.max()),
} for nome, (g, s, f) in cenarios.items()]).set_index("Cenário")

In [ ]:
fig, axs = plt.subplots(2, 3, figsize=(13, 8))
for ax, (nome, (g, s, f)) in zip(axs.ravel(), cenarios.items()):
    mostrar(g, None, None, s, f, nome, ax)      # só o mapa, sem caminho
fig.suptitle("Cenários (castanho mais escuro = terreno mais caro, preto = obstáculo)", y=1.0)
fig.tight_layout()
plt.show()

## 2. Validação com o Anexo A

Antes de qualquer experiência, confirmamos que o Dijkstra dá os custos ótimos do enunciado.

In [ ]:
esperado = {                                   # (4 direções, 8 direções)
    "A labirinto":   (17, 17),
    "B terrenos":    (20, 18.83),
    "C sem caminho": (math.inf, math.inf),
}
linhas = []
for nome, (e4, e8) in esperado.items():
    g, s, f = cenarios[nome]
    c4 = dijkstra(g, s, f, 4)[1]
    c8 = dijkstra(g, s, f, 8)[1]
    ok = math.isclose(c4, e4, abs_tol=0.01) and math.isclose(c8, e8, abs_tol=0.01)
    linhas.append({"Cenário": nome,
                   "4 dir. obtido": c4, "4 dir. esperado": e4,
                   "8 dir. obtido": round(c8, 2), "8 dir. esperado": e8,
                   "OK": "✓" if ok else "✗"})
pd.DataFrame(linhas).set_index("Cenário")

## 3. Dijkstra vs A*

Corremos Dijkstra e A* (com três heurísticas) em todos os cenários, em 4 e 8 direções.
Cada tempo é a **média de 5 execuções**. A optimalidade compara-se com o custo do Dijkstra.

In [ ]:
# (algoritmo, heurística, peso w, função). O Dijkstra tem de ser o primeiro.
configs = [
    ("Dijkstra", "-",          1, lambda g, s, f, d: dijkstra(g, s, f, d)),
    ("A*", "Manhattan",        1, lambda g, s, f, d: astar(g, s, f, d, manhattan)),
    ("A*", "Euclidiana",       1, lambda g, s, f, d: astar(g, s, f, d, euclidiana)),
    ("A*", "Chebyshev",        1, lambda g, s, f, d: astar(g, s, f, d, chebyshev)),
]
df = correr(cenarios, configs, repeticoes=5)
df.to_csv("../results/tables/principal.csv", index=False)
print(f"{len(df)} experiências concluídas")
df.round(3)

In [ ]:
t4 = (df[df.direcoes == 4]
      .pivot_table(index="cenario", columns=["algoritmo", "heuristica"], values="expandidos"))
(t4.rename_axis("Cenário")
   .style.format("{:.0f}")
   .background_gradient(cmap="Greens_r", axis=1)       # verde = menos nós
   .set_caption("Nós expandidos (4 direções)"))

In [ ]:
d = df[df.direcoes == 4]
dij = d[d.algoritmo == "Dijkstra"].set_index("cenario")["expandidos"]
ganho = {}
for h in ["Manhattan", "Euclidiana", "Chebyshev"]:
    a = d[d.heuristica == h].set_index("cenario")["expandidos"]
    ganho[h] = 100 * (1 - a / dij)                      # % de nós poupados
(pd.DataFrame(ganho).rename_axis("Cenário")
   .style.format("{:.1f}%")
   .set_caption("Nós poupados pelo A* face ao Dijkstra (4 direções)"))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.5))
grafico_expandidos(df, 4, ax)
fig.tight_layout()
fig.savefig("../results/figures/expandidos_4dir.png", dpi=150)
plt.show()

In [ ]:
g, s, f = cenarios["Labirinto 51x51"]
fig, axs = plt.subplots(1, 2, figsize=(12, 6))
for ax, (nome, res) in zip(axs, [("Dijkstra", dijkstra(g, s, f, 4)),
                                 ("A* (Manhattan)", astar(g, s, f, 4, manhattan))]):
    mostrar(g, res[0], res[4], s, f, f"{nome}: custo {res[1]:.0f}, {res[2]} nós expandidos", ax)
fig.suptitle("Azul = nós explorados · Vermelho = caminho encontrado")
fig.tight_layout()
fig.savefig("../results/figures/labirinto_caminhos.png", dpi=150)
plt.show()

In [ ]:
g, s, f = cenarios["Custos variados"]
res = dijkstra(g, s, f, 4)
fig, ax = plt.subplots(figsize=(10, 5))
mostrar(g, res[0], res[4], s, f,
        f"Custos variados: custo {res[1]:.0f} em {len(res[0]) - 1} passos (contorna a barreira de custo 9)", ax)
fig.savefig("../results/figures/custos_variados.png", dpi=150)
plt.show()

## 4. Heurísticas e modelo de movimento

| Heurística | 4 direções | 8 direções |
|---|---|---|
| Manhattan | admissível e consistente | **não admissível** |
| Euclidiana | admissível e consistente | admissível e consistente |
| Chebyshev | admissível e consistente | admissível e consistente |

**Porquê?** Em 8 direções, uma diagonal avança 2 unidades de Manhattan por um custo de √2,
por isso o Manhattan sobrestima o custo real e o A* pode devolver caminhos não ótimos.
A tabela seguinte mostra o desvio ao ótimo em 8 direções.

In [ ]:
d8 = df[(df.direcoes == 8) & (df.algoritmo == "A*")]
(d8.pivot_table(index="cenario", columns="heuristica", values="gap_pct")
   .rename_axis("Cenário")
   .style.format("{:.2f}%")
   .set_caption("Desvio ao custo ótimo, A* em 8 direções (0% = ótimo)"))

## 5. Heurística não admissível: Weighted A*

Multiplicamos a heurística por um peso **w > 1**. Com w = 1 o A* é ótimo; com w maior
expande menos nós, mas pode devolver caminhos mais caros. Medimos o compromisso.

In [ ]:
pesos = [1, 1.2, 1.5, 2, 3, 5]
# (lambda w: ...)(w) fixa o valor de w em cada função
configs_w = [("Dijkstra", "-", 1, lambda g, s, f, d: dijkstra(g, s, f, d))] + [
    ("A*", "Manhattan", w,
     (lambda w: lambda g, s, f, d: astar(g, s, f, d, manhattan, w))(w))
    for w in pesos
]
grandes = {k: cenarios[k] for k in ("Labirinto 51x51", "Aberto 100x100", "Custos variados")}
df_w = correr(grandes, configs_w, direcoes_lista=(4,), repeticoes=3)
df_w.to_csv("../results/tables/pesos.csv", index=False)

(df_w[df_w.algoritmo == "A*"][["cenario", "w", "custo", "gap_pct", "expandidos", "tempo_ms"]]
   .rename(columns={"cenario": "Cenário", "gap_pct": "Desvio (%)",
                    "expandidos": "Nós", "tempo_ms": "Tempo (ms)"})
   .round(2).set_index(["Cenário", "w"]))

In [ ]:
for nome in grandes:
    fig = grafico_peso(df_w, nome, "Manhattan", 4)
    fig.savefig(f"../results/figures/peso_{nome.split()[0].lower()}.png", dpi=150)
    plt.show()

## 6. Extensão: Weighted A*

A secção 5 constitui a extensão escolhida: estudo sistemático do compromisso entre
qualidade do caminho e rapidez ao variar o peso da heurística.

*(Se o grupo implementar outra extensão, por exemplo o RMI Ciber-Rato, descrevê-la aqui.)*

## 7. Conclusões

- **Quando compensa o A\*:** *(ex.: espaço aberto, poupa X% dos nós; preencher com os valores obtidos)*
- **Quando compensa pouco:** *(ex.: labirintos com muitos becos sem saída; a heurística ajuda menos)*
- **Heurística e movimento:** *(Manhattan deixa de ser ótimo em 8 direções)*
- **Weighted A\*:** *(a partir de que w o caminho deixa de ser ótimo e quanto se ganha em velocidade)*
- **Limitações:** *(memória em grelhas grandes, sensibilidade à heurística)*
- **O que não esperávamos:** *(preencher)*

---
*Declaração de uso de IA generativa: ver `report/declaracao_ia.md`.*